In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 1. Prepare Data (Consistent with your friend's split)
data = load_wine()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

In [ ]:
# 2. Define the MLP Architecture
class WineMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(WineMLP, self).__init__()
        # First Linear Layer: 13 inputs -> hidden_size outputs
        self.fc1 = nn.Linear(input_size, hidden_size)
        # Activation Function
        self.relu = nn.ReLU()
        # Second Linear Layer: hidden_size -> 3 outputs (one for each class)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [ ]:
# 3. Initialize Model, Loss, and Optimizer
input_dim = 13
hidden_dim = 16  # You can experiment with this (e.g., 8, 32)
output_dim = 3
learning_rate = 0.01
epochs = 100

model = WineMLP(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss() # Standard for multiclass
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# 4. Training Loop
train_losses = []
test_losses = []

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    optimizer.zero_grad()           # Clear old gradients
    outputs = model(X_train_t)      # Forward pass
    loss = criterion(outputs, y_train_t)
    loss.backward()                 # Backward pass (calculates gradients)
    optimizer.step()                # Update weights
    
    train_losses.append(loss.item())
    
    # --- Evaluation Phase ---
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_t)
        test_loss = criterion(test_outputs, y_test_t)
        test_losses.append(test_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}")


In [ ]:
# 5. Final Accuracy Calculation
model.eval()
with torch.no_grad():
    train_preds = torch.argmax(model(X_train_t), dim=1)
    test_preds = torch.argmax(model(X_test_t), dim=1)
    
    train_acc = (train_preds == y_train_t).float().mean()
    test_acc = (test_preds == y_test_t).float().mean()

print(f"\nFinal Accuracy (Train): {train_acc.item()*100:.2f}%")
print(f"Final Accuracy (Test): {test_acc.item()*100:.2f}%")

In [ ]:
# 6. Plotting the Loss Curves (As required by task)
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Testing Loss', linestyle='--')
plt.title('MLP Training & Testing Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
Epoch [10/100], Train Loss: 1.0644, Test Loss: 1.0356
Epoch [20/100], Train Loss: 1.0158, Test Loss: 0.9867
Epoch [30/100], Train Loss: 0.9698, Test Loss: 0.9404
Epoch [40/100], Train Loss: 0.9259, Test Loss: 0.8962
Epoch [50/100], Train Loss: 0.8839, Test Loss: 0.8536
Epoch [60/100], Train Loss: 0.8437, Test Loss: 0.8127
Epoch [70/100], Train Loss: 0.8049, Test Loss: 0.7732
Epoch [80/100], Train Loss: 0.7677, Test Loss: 0.7351
Epoch [90/100], Train Loss: 0.7321, Test Loss: 0.6983
Epoch [100/100], Train Loss: 0.6982, Test Loss: 0.6630

Final Accuracy (Train): 87.32%
Final Accuracy (Test): 94.44%

![MLP](MLP.png)